# InfoPrune — Phase 1 (consolidated, runs top-to-bottom in a fresh Kaggle session)

Replaces both `phase1-infoprune.ipynb` and `consolidated-infoprune.ipynb`. **Restart the kernel and
run every cell in order** — nothing depends on state from the Phase 0 notebooks anymore.

**What was actually breaking** (none of it was a Python syntax error — every cell in both notebooks
parses; the failures were state/ordering problems):

| Problem | Where | Fix here |
|---|---|---|
| `FileNotFoundError` loading the "pristine" `modeling_llama.py` — FastV wasn't cloned in that session; and even with the file present, `exec_module` on it fails because the file uses relative imports (`from ...activations import ...`) | consolidated §1 | No file reloading. The original `fastv_forward` is stashed on the class once, with an assert that it really is the library's function |
| `_ORIGINAL_FASTV_FORWARD` captured from the live class, so re-running the cell after `enable_infoprune()` stored InfoPrune as the "original" | phase1 cell 14 | Same stash + assert |
| `importlib.reload(modeling_llama)` creates *new* classes mid-session; the old patch/originals then point at stale objects | phase1 cell 26 | Removed. No enable/disable toggling at all: the method is chosen by `fastv_config["method"]`, and `model.config.fastv_config` is swapped at runtime on **one** loaded model |
| Relies on `MODEL_ID`, `processor`, `bnb_config`, `img_dir`, `all_questions`, `requests`, `LlavaForConditionalGeneration` from another notebook | consolidated (all sections) | Self-contained |
| `re` never imported (`parse_yes_no` fallback → `NameError`) | phase1 cell 24 | Imported |
| Multi-GPU device fixes (`text_token_idx.to(...)`, `image_scores.to(...)`) present in phase1 but dropped in consolidated → device-mismatch error on 2×T4 | consolidated §1 | Restored |

**Correctness fixes to the method itself** (these change numbers — see the notes in §5):
1. **Root cause of the NaN traces:** `eps=1e-8` underflows to exactly `0` in fp16, so any attention weight that is exactly 0 gives `0 * log(0) = NaN`. The clamp didn't fix that. Entropy is now computed in fp32 with `torch.special.entr`. Dropping NaN traces was also biased: it removed exactly the *peaky, low-entropy* layers, and at inference a NaN layer got percentile 0, so it could never be chosen.
2. **Calibration used the POPE evaluation images** (test-set leakage) and a different prompt ("Describe this image.") than evaluation, even though the text-token filter depends on the prompt. Calibration now uses disjoint images with benchmark-style questions, per benchmark.
3. **Calibration now runs through the exact same forward function as inference** (`selection="calibrate"`), so the text-token selection and entropy code can't drift apart.

## 1. Environment check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB, cc {p.major}.{p.minor}")

## 2. Install FastV's patched transformers (4.39.0.dev0)
If `transformers` was already imported in this kernel before this cell ran, restart the kernel after it finishes.

In [ ]:
%%bash
cd /kaggle/working
if [ ! -d "FastV" ]; then
  git clone --depth 1 https://github.com/pkunlp-icler/FastV.git
fi
cd FastV/src/FastV/llava-hf/transformers
pip install -e . --no-deps -q
pip install -q "tokenizers<0.19,>=0.14" pillow accelerate bitsandbytes sentencepiece
python -c "import transformers; print('transformers version:', transformers.__version__)"

## 3. Imports, paths, constants

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import bisect, gc, json, random, re, string, time, urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import requests
import torch
from PIL import Image

import transformers
from transformers import (AutoTokenizer, CLIPImageProcessor, LlavaProcessor,
                          LlavaForConditionalGeneration, BitsAndBytesConfig)
from transformers.modeling_outputs import BaseModelOutputWithPast
import transformers.models.llama.modeling_llama as modeling_llama

print("transformers", transformers.__version__)
assert hasattr(modeling_llama.LlamaModel, "fastv_forward"), (
    "This is not FastV's patched transformers. Run Section 2, then restart the kernel.")

WORK = Path("/kaggle/working")
DATA, RESULTS, CACHE = WORK / "data", WORK / "results", WORK / "cache"
for p in (DATA, RESULTS, CACHE):
    p.mkdir(parents=True, exist_ok=True)

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
IMAGE_START, IMAGE_LEN = 5, 576        # "USER: <image>..." -> image features occupy positions 5..580
N_LAYERS, D_MODEL, D_FFN = 32, 4096, 11008

## 4. Load processor + model **once**

The FastV fork's `LlavaForConditionalGeneration.forward` reads `self.config.fastv_config` on every call,
so switching method = assigning `model.config.fastv_config`. No reloading between baseline / FastV /
InfoPrune runs (saves a load per run and removes the OOM churn).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
image_processor = CLIPImageProcessor.from_pretrained(MODEL_ID)
processor = LlavaProcessor(image_processor=image_processor, tokenizer=tokenizer)
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

if "model" not in globals():
    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, attn_implementation="eager", device_map="auto")
    model.eval()
model.config.fastv_config = None
print("Loaded. Devices:", sorted({str(p.device) for p in model.parameters()}))

In [ ]:
def set_method(cfg):
    """cfg=None -> plain baseline forward; otherwise routed through LlamaModel.fastv_forward."""
    model.config.fastv_config = cfg

def build_inputs(image, question, suffix=""):
    prompt = f"USER: <image>\n{question}{suffix}\nASSISTANT:"
    inputs = processor(prompt, image, return_tensors="pt").to(model.device)
    pos = (inputs["input_ids"][0] == model.config.image_token_index).nonzero().flatten()
    assert pos.numel() == 1 and pos.item() == IMAGE_START, f"image token at {pos.tolist()}, expected [{IMAGE_START}]"
    return inputs

@torch.no_grad()
def generate(inputs, max_new_tokens):
    return model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=False)

def decode_answer(out):
    return processor.decode(out[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

def free_mem():
    gc.collect(); torch.cuda.empty_cache()

# Prefill FLOPs, FastV paper's formula: per layer 4nd^2 + 2n^2 d + 2ndm.
def layer_flops(n):
    return 4 * n * D_MODEL**2 + 2 * n * n * D_MODEL + 2 * n * D_MODEL * D_FFN

def relative_prefill_flops(seq_len, k_full, r):
    """k_full = number of decoder layers that run on the full sequence (FastV's K)."""
    if k_full is None or k_full >= N_LAYERS:
        return 1.0
    n_pruned = seq_len - round(IMAGE_LEN * r)
    return (k_full * layer_flops(seq_len) + (N_LAYERS - k_full) * layer_flops(n_pruned)) / (N_LAYERS * layer_flops(seq_len))

## 5. The InfoPrune module

**Scoring note.** The old score was `percentile_l( H_data(l) / H_img(l) )` computed within each layer's
calibration distribution. `H_data(l)` is a constant per layer, and a within-layer percentile is
invariant to a monotone transform, so `H_data(l)` cancels exactly. The score is (and was) simply
*"fraction of calibration images whose attention at layer l was more diffuse than this image's."*
`focus_score` below computes that directly — same ranking as before, one fewer thing to reason
about. `H_data(l)` is still computed and reported as a per-layer diagnostic.

**Selection modes**
- `argmax` — your previous method: score layers `floor..ceiling`, rank image tokens using the
  highest-scoring layer's attention, **prune after `ceiling`**. Since pruning always happens after
  layer 16, this is FastV-K=17-equivalent compute, not K=2. The fair control is FastV K=17 (run in §8).
- `threshold` — the plan's per-instance `K*`: prune right after the first layer whose score ≥ τ
  (falls back to argmax at the ceiling). This is the mode that actually saves compute; τ is set from
  calibration to hit a target mean `K*`.
- `calibrate` — no pruning; records `H_img(l)` and the image-attention vector per layer.

`k_equiv = prune_after + 1` is reported in FastV's K convention (FastV K=2 prunes after layer 1).

In [ ]:
TRACE = {}
def trace_reset():
    TRACE.clear(); TRACE["calls"] = 0


def relevant_text_token_idx(embeds, img_s, img_n):
    """SparseVLM 'text raters': image-patch x text-token similarity on the pre-layer-0 embeddings,
    softmax over text, mean over patches, keep tokens above the mean. Returns absolute positions."""
    text_s = img_s + img_n
    v = embeds[0, img_s:text_s].float()
    t = embeds[0, text_s:].float()
    if t.shape[0] == 0:
        return torch.empty(0, dtype=torch.long, device=embeds.device)
    rel = (v @ t.T).softmax(dim=1).mean(dim=0)                 # (T,)
    return torch.where(rel > rel.mean())[0] + text_s


def image_attention_entropy(attn, img_s, img_n, text_idx):
    """attn: (1, heads, q, k) from one layer. Query = mean over heads and relevant text rows
    (last token if none). Computed in fp32: in fp16, eps=1e-8 rounds to 0 and 0*log(0)=NaN."""
    a = attn[0]
    if text_idx is not None and text_idx.numel() > 0:
        rows = a[:, text_idx.to(a.device), :]
    else:
        rows = a[:, -1:, :]
    p = rows.float().mean(dim=(0, 1))[img_s:img_s + img_n].clamp_min(0)
    p = p / p.sum().clamp_min(1e-12)
    return torch.special.entr(p).sum().item(), p


def focus_score(h, sorted_calib_h):
    """Fraction of calibration images with higher entropy (more diffuse attention) at this layer.
    Identical to the old percentile of H_data(l)/H_img, since H_data(l) cancels."""
    n = len(sorted_calib_h)
    return (n - bisect.bisect_right(sorted_calib_h, h)) / n


def infoprune_forward(self, input_ids=None, attention_mask=None, position_ids=None, past_key_values=None,
                      inputs_embeds=None, use_cache=None, output_attentions=None, output_hidden_states=None,
                      return_dict=None, cache_position=None, fastv_config=None):
    cfg = fastv_config
    img_s, img_n = cfg["image_token_start_index"], cfg["image_token_length"]
    lo, hi = cfg["layer_floor"], cfg["layer_ceiling"]
    mode = cfg["selection"]                                 # "argmax" | "threshold" | "calibrate"
    r = cfg.get("fastv_r", 0.5)
    calib_h, tau = cfg.get("calib_h"), cfg.get("threshold")
    assert mode in ("argmax", "threshold", "calibrate"), mode
    assert mode == "calibrate" or calib_h is not None, "run calibration first"
    assert mode != "threshold" or tau is not None, "threshold mode needs cfg['threshold']"

    use_cache = use_cache if use_cache is not None else self.config.use_cache
    assert not use_cache, "InfoPrune supports use_cache=False only."
    output_hidden_states = output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
    return_dict = return_dict if return_dict is not None else self.config.use_return_dict

    if inputs_embeds is None:
        inputs_embeds = self.embed_tokens(input_ids)
    seq_len = inputs_embeds.shape[1]
    if cache_position is None:
        cache_position = torch.arange(seq_len, device=inputs_embeds.device)
    if position_ids is None:
        position_ids = cache_position.unsqueeze(0)

    mask = self._update_causal_mask(attention_mask, inputs_embeds, 0)
    hidden_states = inputs_embeds
    all_hidden = () if output_hidden_states else None
    is_prefill = seq_len > 1
    text_idx = relevant_text_token_idx(inputs_embeds, img_s, img_n) if is_prefill else None

    h_trace, scores, vecs = {}, {}, {}
    pruned, chosen, prune_after = False, None, None

    for layer_idx, layer in enumerate(self.layers):
        if output_hidden_states:
            all_hidden += (hidden_states,)
        in_window = is_prefill and not pruned and lo <= layer_idx <= hi
        out = layer(hidden_states, attention_mask=mask, position_ids=position_ids, past_key_value=None,
                    output_attentions=in_window, use_cache=False, cache_position=cache_position)
        hidden_states = out[0]
        if not in_window:
            continue

        h, p = image_attention_entropy(out[1], img_s, img_n, text_idx)
        del out
        h_trace[layer_idx] = h
        if mode == "calibrate":
            vecs[layer_idx] = p.cpu()
            continue

        scores[layer_idx] = focus_score(h, calib_h[layer_idx])
        vecs[layer_idx] = p
        if mode == "threshold" and scores[layer_idx] >= tau:
            chosen = layer_idx
        elif layer_idx == hi:
            chosen = max(scores, key=scores.get)
        if chosen is None:
            continue

        # Prune after this layer, FastV mechanics, ranking image tokens by the chosen layer's attention.
        prune_after = layer_idx
        dev = hidden_states.device
        top = vecs[chosen].to(dev).topk(round(img_n * (1 - r))).indices + img_s
        keep_idx = torch.cat((torch.arange(img_s, device=dev), top,
                              torch.arange(img_s + img_n, seq_len, device=dev))).sort().values
        hidden_states = hidden_states[:, keep_idx, :]
        position_ids = keep_idx.unsqueeze(0)
        mask = self._update_causal_mask(None, hidden_states, 0)
        cache_position = cache_position[: keep_idx.shape[0]]
        pruned = True
        vecs.clear()

    hidden_states = self.norm(hidden_states)
    if output_hidden_states:
        all_hidden += (hidden_states,)

    if is_prefill and TRACE.get("calls", 0) == 0:     # record the first (prefill) call of each generate()
        TRACE.update(mode=mode, chosen_layer=chosen, prune_after=prune_after,
                     k_equiv=None if prune_after is None else prune_after + 1,
                     scores={l: round(s, 4) for l, s in scores.items()}, h_img=dict(h_trace),
                     seq_before=seq_len, seq_after=hidden_states.shape[1])
        if mode == "calibrate":
            TRACE["img_attn"] = vecs
    TRACE["calls"] = TRACE.get("calls", 0) + 1

    if not return_dict:
        return tuple(v for v in (hidden_states, None, all_hidden) if v is not None)
    return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=None,
                                   hidden_states=all_hidden, attentions=None)


# ---- Install once. Safe to re-run this cell after editing any function above. ----
_LM = modeling_llama.LlamaModel
if not hasattr(_LM, "_fastv_forward_original"):
    _LM._fastv_forward_original = _LM.fastv_forward
assert getattr(_LM._fastv_forward_original, "__module__", "") == modeling_llama.__name__, (
    "LlamaModel.fastv_forward was already monkeypatched by an older notebook in this kernel. "
    "Restart the kernel and run this notebook from the top.")

def _fastv_dispatch(self, *args, fastv_config=None, **kwargs):
    if fastv_config is not None and fastv_config.get("method") == "infoprune":
        return infoprune_forward(self, *args, fastv_config=fastv_config, **kwargs)   # looked up at call time
    return _LM._fastv_forward_original(self, *args, fastv_config=fastv_config, **kwargs)

_LM.fastv_forward = _fastv_dispatch


def fastv_cfg(k, r=0.5):
    return {"method": "fastv", "use_fastv": True, "fastv_k": k, "fastv_r": r,
            "image_token_start_index": IMAGE_START, "image_token_length": IMAGE_LEN}

def infoprune_cfg(calib, selection, r=0.5, threshold=None):
    return {"method": "infoprune", "selection": selection, "fastv_r": r, "threshold": threshold,
            "image_token_start_index": IMAGE_START, "image_token_length": IMAGE_LEN,
            "layer_floor": calib["layer_floor"], "layer_ceiling": calib["layer_ceiling"],
            "calib_h": calib["calib_h"]}

print("InfoPrune installed; original fastv_forward stashed:", _LM._fastv_forward_original.__qualname__)

## 6. POPE data — evaluation subset (identical to Phase 0) + a **disjoint** calibration set

In [ ]:
%%bash
mkdir -p /kaggle/working/data && cd /kaggle/working/data
[ -d POPE ] || git clone --depth 1 https://github.com/AoiDragon/POPE.git

In [ ]:
pope_dir = DATA / "POPE/output/coco"
POPE_SPLITS = ["coco_pope_random.json", "coco_pope_popular.json", "coco_pope_adversarial.json"]
N_PER_SPLIT = 150
N_CALIB = 150
coco_dir = DATA / "coco_val2014_subset"
coco_dir.mkdir(exist_ok=True)

# Same sampling code/seed as Phase 0 -> same 450 questions, so numbers stay comparable.
random.seed(0)
pope_lines, pope_eval = {}, []
for split in POPE_SPLITS:
    lines = [json.loads(l) for l in open(pope_dir / split) if l.strip()]
    pope_lines[split] = lines
    images = sorted(set(l["image"] for l in lines))
    random.shuffle(images)
    keep = set(images[: max(1, N_PER_SPLIT // 6)])
    for l in [l for l in lines if l["image"] in keep][:N_PER_SPLIT]:
        pope_eval.append({"id": f"{split.split('_')[2].split('.')[0]}:{l['question_id']}", "split": split,
                          "image_path": str(coco_dir / l["image"]), "question": l["text"], "label": l["label"]})

# Calibration: POPE-style questions on images that never appear in the eval subset.
eval_imgs = {Path(q["image_path"]).name for q in pope_eval}
by_img = defaultdict(list)
for lines in pope_lines.values():
    for l in lines:
        if l["image"] not in eval_imgs:
            by_img[l["image"]].append(l)
rng = random.Random(1)
calib_img_names = rng.sample(sorted(by_img), min(N_CALIB, len(by_img)))
pope_calib = [{"image_path": str(coco_dir / n), "question": rng.choice(by_img[n])["text"]} for n in calib_img_names]

def fetch(fname):
    dest = coco_dir / fname
    if not dest.exists():
        urllib.request.urlretrieve(f"http://images.cocodataset.org/val2014/{fname}", dest)

need = sorted(eval_imgs | set(calib_img_names))
with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(fetch, need))
print(f"POPE eval: {len(pope_eval)} questions / {len(eval_imgs)} images | calibration: {len(pope_calib)} images (disjoint)")

## 7. Calibration (cached to disk; re-running is free)

In [ ]:
LAYER_FLOOR, LAYER_CEILING = 2, 16   # set LAYER_FLOOR=1 so threshold mode can reach FastV's K=2 compute

@torch.no_grad()
def calibrate(items, tag, layer_floor=LAYER_FLOOR, layer_ceiling=LAYER_CEILING, suffix="", force=False):
    path = CACHE / f"calib_{tag}_L{layer_floor}-{layer_ceiling}_n{len(items)}.json"
    if path.exists() and not force:
        c = json.load(open(path))
        c["calib_h"] = {int(k): v for k, v in c["calib_h"].items()}
        c["H_data"] = {int(k): v for k, v in c["H_data"].items()}
        c["traces"] = [{int(k): v for k, v in t.items()} for t in c["traces"]]
        print("Loaded cached calibration:", path.name)
        return c

    layers = list(range(layer_floor, layer_ceiling + 1))
    set_method({"method": "infoprune", "selection": "calibrate", "fastv_r": 0.0,
                "image_token_start_index": IMAGE_START, "image_token_length": IMAGE_LEN,
                "layer_floor": layer_floor, "layer_ceiling": layer_ceiling})
    sums, traces = {l: torch.zeros(IMAGE_LEN) for l in layers}, []
    try:
        for i, it in enumerate(items):
            inputs = build_inputs(Image.open(it["image_path"]).convert("RGB"), it["question"], suffix)
            trace_reset()
            model(**inputs, use_cache=False)                   # one prefill pass, same code path as inference
            traces.append(dict(TRACE["h_img"]))
            for l in layers:
                sums[l] += TRACE["img_attn"][l]
            if (i + 1) % 50 == 0:
                print(f"  calibrated {i+1}/{len(items)}")
    finally:
        set_method(None)
        free_mem()

    n_nan = sum(any(np.isnan(v) for v in t.values()) for t in traces)
    assert n_nan == 0, f"{n_nan} NaN traces -- should be impossible with fp32 entropy; investigate"
    H_data = {l: torch.special.entr(sums[l] / sums[l].sum()).sum().item() for l in layers}
    calib_h = {l: sorted(t[l] for t in traces) for l in layers}
    c = {"tag": tag, "layer_floor": layer_floor, "layer_ceiling": layer_ceiling,
         "H_data": H_data, "calib_h": calib_h, "traces": traces}
    json.dump(c, open(path, "w"))
    return c

def print_calibration(c):
    print(f"{'layer':>5} {'H_data':>8} {'H_img median':>13} {'H_img IQR':>18}")
    for l in range(c["layer_floor"], c["layer_ceiling"] + 1):
        h = np.array(c["calib_h"][l])
        print(f"{l:>5} {c['H_data'][l]:>8.4f} {np.median(h):>13.4f}   [{np.percentile(h,25):.3f}, {np.percentile(h,75):.3f}]")
    print(f"(max possible entropy over {IMAGE_LEN} tokens = {np.log(IMAGE_LEN):.4f})")

calib_pope = calibrate(pope_calib, "pope")
print_calibration(calib_pope)

### Choosing τ for threshold mode
Simulated on the calibration traces (in-sample, so treat as a budget knob, not an accuracy estimate).
`TARGET_MEAN_K` is the average FastV-equivalent K you want to spend; §8 adds a FastV run at the
same (rounded) K as the matched-compute control.

In [ ]:
TARGET_MEAN_K = 4

def simulate_threshold(c, tau):
    lo, hi = c["layer_floor"], c["layer_ceiling"]
    ks = []
    for t in c["traces"]:
        stop = hi
        for l in range(lo, hi + 1):
            if focus_score(t[l], c["calib_h"][l]) >= tau:
                stop = l
                break
        ks.append(stop + 1)
    return np.array(ks)

def pick_threshold(c, target_mean_k, show=True):
    grid = [round(x, 2) for x in np.arange(0.50, 1.0001, 0.01)]
    means = {tau: simulate_threshold(c, tau).mean() for tau in grid}
    tau = min(grid, key=lambda t: abs(means[t] - target_mean_k))
    if show:
        for t in grid[::5]:
            print(f"  tau={t:.2f}  mean K*={means[t]:.2f}")
    ks = simulate_threshold(c, tau)
    print(f"-> tau={tau:.2f}: mean K*={ks.mean():.2f}, K* distribution {dict(sorted(Counter(ks.tolist()).items()))}")
    return tau, ks.mean()

tau_pope, meanK_pope = pick_threshold(calib_pope, TARGET_MEAN_K)

## 8. Smoke test — one image through every method
Checks: coherent answers, InfoPrune removes 288 tokens, InfoPrune's logits actually differ from baseline
(i.e. the patch is live), and the chosen layer varies across images.

In [ ]:
demo_urls = ["http://images.cocodataset.org/val2017/000000039769.jpg",
             "http://images.cocodataset.org/val2017/000000000139.jpg",
             "http://images.cocodataset.org/val2017/000000000285.jpg",
             "http://images.cocodataset.org/val2017/000000000724.jpg",
             "http://images.cocodataset.org/val2017/000000001000.jpg"]
demo_imgs = [Image.open(requests.get(u, stream=True).raw).convert("RGB") for u in demo_urls]
q = "What is shown in this image? Answer in one sentence."

smoke_methods = {"baseline": None, "fastv_K2": fastv_cfg(2),
                 "infoprune_argmax": infoprune_cfg(calib_pope, "argmax"),
                 "infoprune_threshold": infoprune_cfg(calib_pope, "threshold", threshold=tau_pope)}
inputs = build_inputs(demo_imgs[0], q)
last_logits = {}
for name, cfg in smoke_methods.items():
    set_method(cfg)
    trace_reset()
    with torch.no_grad():
        last_logits[name] = model(**inputs, use_cache=False).logits[0, -1].float().cpu()
    trace_reset()
    ans = decode_answer(generate(inputs, 32))
    extra = f" | K*={TRACE['k_equiv']} seq {TRACE['seq_before']}->{TRACE['seq_after']}" if "infoprune" in name else ""
    print(f"{name:<20} {ans}{extra}")
set_method(None)
for name in list(smoke_methods)[1:]:
    print(f"max |logit diff| vs baseline, {name}: {(last_logits[name] - last_logits['baseline']).abs().max():.4f}")

print("\nPer-image choice (threshold mode):")
set_method(smoke_methods["infoprune_threshold"])
for u, img in zip(demo_urls, demo_imgs):
    trace_reset()
    generate(build_inputs(img, q), 1)
    print(f"  {u.split('/')[-1]}: chosen layer {TRACE['chosen_layer']}, K*={TRACE['k_equiv']}, scores {TRACE['scores']}")
set_method(None)
free_mem()

## 9. Evaluation harness (resumable)
Each (benchmark, method) writes one JSON line per example to `/kaggle/working/results/`. If the kernel
dies, re-run and it resumes. **If you change the method code or calibration, delete the affected
`.jsonl` files** (the name encodes the main knobs, not the code).

In [ ]:
def run_method(bench, name, cfg, items, max_new_tokens, suffix, score_fn):
    path = RESULTS / f"{bench}__{name}.jsonl"
    done = {}
    if path.exists():
        for line in open(path):
            rec = json.loads(line)
            done[rec["id"]] = rec
    todo = [it for it in items if it["id"] not in done]
    print(f"[{bench}/{name}] {len(done)} cached, {len(todo)} to run")
    r = cfg["fastv_r"] if cfg else 0.0
    set_method(cfg)
    try:
        with open(path, "a") as f:
            for i, it in enumerate(todo):
                inputs = build_inputs(Image.open(it["image_path"]).convert("RGB"), it["question"], suffix)
                seq_len = inputs["input_ids"].shape[1] - 1 + IMAGE_LEN
                trace_reset()
                t0 = time.time()
                ans = decode_answer(generate(inputs, max_new_tokens))
                dt = time.time() - t0
                if cfg is None:
                    k = N_LAYERS
                elif cfg["method"] == "fastv":
                    k = cfg["fastv_k"]
                else:
                    k = TRACE.get("k_equiv")
                rec = {"id": it["id"], "answer": ans, "latency_s": round(dt, 3), "k_equiv": k,
                       "rel_flops": round(relative_prefill_flops(seq_len, k, r), 4),
                       "chosen_layer": TRACE.get("chosen_layer"), "scores": TRACE.get("scores")}
                rec.update(score_fn(ans, it))
                f.write(json.dumps(rec) + "\n")
                f.flush()
                done[it["id"]] = rec
                if (i + 1) % 50 == 0:
                    print(f"  {i+1}/{len(todo)}")
    finally:
        set_method(None)
        free_mem()
    return [done[it["id"]] for it in items]


def summarize_compute(recs):
    ks = [r["k_equiv"] for r in recs if r["k_equiv"] is not None]
    return dict(mean_k=float(np.mean(ks)) if ks else float("nan"),
                rel_flops=float(np.mean([r["rel_flops"] for r in recs])))

## 10. POPE comparison
`max_new_tokens=3`: with `use_cache=False` every generated token re-runs the full prefill, and the
yes/no parse only looks at the first word, so 3 tokens gives the same predictions as 10 at ~⅓ the cost.

In [ ]:
def parse_yes_no(text):
    text = text.strip().lower()
    if text.startswith("yes"): return "yes"
    if text.startswith("no"):  return "no"
    m = re.search(r"\b(yes|no)\b", text)
    return m.group(1) if m else "no"

def pope_score(ans, it):
    return {"pred": parse_yes_no(ans), "label": it["label"]}

def pope_metrics(preds, labels):
    tp = sum(p == "yes" and l == "yes" for p, l in zip(preds, labels))
    fp = sum(p == "yes" and l == "no" for p, l in zip(preds, labels))
    tn = sum(p == "no" and l == "no" for p, l in zip(preds, labels))
    fn = sum(p == "no" and l == "yes" for p, l in zip(preds, labels))
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    return dict(accuracy=(tp + tn) / len(labels), precision=prec, recall=rec,
                f1=2 * prec * rec / (prec + rec) if prec + rec else 0.0,
                yes_ratio=preds.count("yes") / len(preds))

R = 0.5
k_match = int(round(meanK_pope))
pope_methods = {
    "baseline": None,
    f"fastv_K2_R{R}": fastv_cfg(2, R),
    f"fastv_K{LAYER_CEILING+1}_R{R}": fastv_cfg(LAYER_CEILING + 1, R),        # same prune point as argmax
    f"infoprune_argmax_L{LAYER_FLOOR}-{LAYER_CEILING}_R{R}": infoprune_cfg(calib_pope, "argmax", R),
    f"infoprune_thr{tau_pope:.2f}_L{LAYER_FLOOR}-{LAYER_CEILING}_R{R}": infoprune_cfg(calib_pope, "threshold", R, tau_pope),
    f"fastv_K{k_match}_R{R}": fastv_cfg(k_match, R),                          # matched to threshold's mean K*
}

pope_runs = {name: run_method("pope", name, cfg, pope_eval, 3, "", pope_score)
             for name, cfg in pope_methods.items()}

In [ ]:
def pope_table(runs):
    base = {r["id"]: r["pred"] for r in runs["baseline"]}
    print(f"{'method':<40}{'acc':>7}{'F1':>7}{'yes%':>7}{'mean K':>8}{'FLOPs':>8}{'agree':>8}")
    for name, recs in runs.items():
        m = pope_metrics([r["pred"] for r in recs], [r["label"] for r in recs])
        c = summarize_compute(recs)
        agree = np.mean([base[r["id"]] == r["pred"] for r in recs])
        print(f"{name:<40}{m['accuracy']*100:>6.1f}%{m['f1']*100:>6.1f}%{m['yes_ratio']*100:>6.1f}%"
              f"{c['mean_k']:>8.2f}{c['rel_flops']*100:>7.1f}%{agree*100:>7.1f}%")

pope_table(pope_runs)

print("\nPer-split accuracy:")
for name, recs in pope_runs.items():
    accs = {s: np.mean([r["pred"] == r["label"] for r in recs if r["id"].startswith(s)])
            for s in ("random", "popular", "adversarial")}
    print(f"  {name:<40}" + "  ".join(f"{s}={a*100:.1f}%" for s, a in accs.items()))

## 11. GQA — data (Kaggle-hosted mirrors), disjoint calibration, evaluation
Prompt suffix is LLaVA-1.5's official GQA prompt (`Answer the question using a single word or phrase.`),
so baseline numbers are comparable to published ones. Scoring reports both exact match (what LLaVA's
eval uses) and a lenient contains-match.

In [ ]:
GQA_Q_FILE = None      # override with a Path if autodetection fails
GQA_IMG_DIR = None     # override with a Path if autodetection fails
N_GQA_EVAL, N_GQA_CALIB = 300, 150
GQA_SUFFIX = "\nAnswer the question using a single word or phrase."

if GQA_Q_FILE is None:
    GQA_Q_FILE = next(Path("/kaggle/input").rglob("testdev_balanced_questions.json"))
gqa_q = json.load(open(GQA_Q_FILE))

def find_image_dir(sample_image_id, root="/kaggle/input"):
    for dirpath, dirnames, filenames in os.walk(root):
        if f"{sample_image_id}.jpg" in filenames:
            return Path(dirpath)
        if len(filenames) > 1000:      # large non-matching image dir: don't descend further
            dirnames[:] = []
    raise FileNotFoundError("GQA images not found under /kaggle/input -- set GQA_IMG_DIR")

qids = list(gqa_q.keys())
random.Random(0).shuffle(qids)          # same order as random.seed(0); random.shuffle(qids)
eval_qids = qids[:N_GQA_EVAL]
if GQA_IMG_DIR is None:
    GQA_IMG_DIR = find_image_dir(gqa_q[eval_qids[0]]["imageId"])
print("GQA questions:", GQA_Q_FILE, "| images:", GQA_IMG_DIR)

def gqa_item(qid):
    q = gqa_q[qid]
    return {"id": qid, "image_path": str(GQA_IMG_DIR / f"{q['imageId']}.jpg"), "question": q["question"],
            "answer": q["answer"], "structural": q.get("types", {}).get("structural"),
            "semantic": q.get("types", {}).get("semantic")}

gqa_eval = [gqa_item(q) for q in eval_qids]
gqa_eval_imgs = {gqa_q[q]["imageId"] for q in eval_qids}
gqa_calib, seen = [], set()
for qid in qids[N_GQA_EVAL:]:
    img_id = gqa_q[qid]["imageId"]
    if img_id in gqa_eval_imgs or img_id in seen:
        continue
    seen.add(img_id)
    gqa_calib.append(gqa_item(qid))
    if len(gqa_calib) == N_GQA_CALIB:
        break
missing = [it["image_path"] for it in gqa_eval + gqa_calib if not Path(it["image_path"]).exists()]
assert not missing, f"{len(missing)} GQA images missing, e.g. {missing[:3]}"
print(f"GQA eval: {len(gqa_eval)} q / {len(gqa_eval_imgs)} images | calibration: {len(gqa_calib)} (disjoint images)")

In [ ]:
calib_gqa = calibrate(gqa_calib, "gqa", suffix=GQA_SUFFIX)
print_calibration(calib_gqa)
tau_gqa, meanK_gqa = pick_threshold(calib_gqa, TARGET_MEAN_K)

In [ ]:
_ARTICLES = {"a", "an", "the"}
def norm_answer(s):
    s = s.lower().strip().translate(str.maketrans("", "", string.punctuation))
    return " ".join(w for w in s.split() if w not in _ARTICLES)

def gqa_score(ans, it):
    p, g = norm_answer(ans), norm_answer(it["answer"])
    return {"pred": p, "gt": g, "exact": p == g, "lenient": p == g or f" {g} " in f" {p} "}

k_match_gqa = int(round(meanK_gqa))
gqa_methods = {
    "baseline": None,
    f"fastv_K2_R{R}": fastv_cfg(2, R),
    f"fastv_K{LAYER_CEILING+1}_R{R}": fastv_cfg(LAYER_CEILING + 1, R),
    f"infoprune_argmax_L{LAYER_FLOOR}-{LAYER_CEILING}_R{R}": infoprune_cfg(calib_gqa, "argmax", R),
    f"infoprune_thr{tau_gqa:.2f}_L{LAYER_FLOOR}-{LAYER_CEILING}_R{R}": infoprune_cfg(calib_gqa, "threshold", R, tau_gqa),
    f"fastv_K{k_match_gqa}_R{R}": fastv_cfg(k_match_gqa, R),
}
gqa_runs = {name: run_method("gqa", name, cfg, gqa_eval, 10, GQA_SUFFIX, gqa_score)
            for name, cfg in gqa_methods.items()}

base = {r["id"]: r["pred"] for r in gqa_runs["baseline"]}
print(f"\n{'method':<40}{'exact':>7}{'lenient':>9}{'mean K':>8}{'FLOPs':>8}{'agree':>8}")
for name, recs in gqa_runs.items():
    c = summarize_compute(recs)
    print(f"{name:<40}{np.mean([r['exact'] for r in recs])*100:>6.1f}%{np.mean([r['lenient'] for r in recs])*100:>8.1f}%"
          f"{c['mean_k']:>8.2f}{c['rel_flops']*100:>7.1f}%{np.mean([base[r['id']] == r['pred'] for r in recs])*100:>7.1f}%")

## 12. Is the adaptivity real? (seed of the Phase 3 analysis)
Chosen-layer distribution per benchmark, and whether it shifts with POPE split / GQA question type.
If every group has the same distribution, the layer choice is image-noise, not task-driven.

In [ ]:
def layer_breakdown(runs, items, group_key):
    groups = {it["id"]: group_key(it) for it in items}
    for name, recs in runs.items():
        if not name.startswith("infoprune"):
            continue
        print(f"\n{name}\n  overall chosen layer: {dict(sorted(Counter(r['chosen_layer'] for r in recs).items()))}")
        by = defaultdict(list)
        for r in recs:
            by[groups[r["id"]]].append(r["chosen_layer"])
        for g, ls in sorted(by.items(), key=lambda kv: str(kv[0])):
            print(f"  {str(g):<14} n={len(ls):<4} mean layer={np.mean(ls):5.2f}  {dict(sorted(Counter(ls).items()))}")

layer_breakdown(pope_runs, pope_eval, lambda it: it["id"].split(":")[0])
if "gqa_runs" in globals():
    layer_breakdown(gqa_runs, gqa_eval, lambda it: it["structural"])

## What to send back
1. The calibration tables (POPE and GQA) and the chosen τ / K* distribution.
2. §8 smoke-test output (answers, logit diffs, per-image layers).
3. The POPE and GQA tables from §10/§11 and the breakdown from §12.

**How to read the tables.** The claim "InfoPrune beats FastV" only holds if it wins *at matched
FLOPs*: compare `infoprune_argmax` against `fastv_K17` (same prune point), and `infoprune_thr…`
against `fastv_K{k_match}`. If argmax ≈ FastV-K17, the layer *selection* isn't adding anything beyond
"prune late"; if threshold beats its matched FastV K, that's the result the paper needs.

**Known limits (unchanged, Phase 2 work):** `use_cache=False` only, so wall-clock latency here is not
meaningful — report FLOPs. The text-token filter uses all tokens after the image (including
"ASSISTANT:" and, during decoding, generated tokens); SparseVLM restricts to the instruction.